# Step 8 — Endpoint Deployment

Deploy the registered model as a real-time inference endpoint.

In [ ]:
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.model_monitor import DataCaptureConfig

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
ENDPOINT_NAME = "symptom-classifier-endpoint"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()
sm_client = boto3.client("sagemaker", region_name=REGION)

# Get model URI
training_jobs = sm_client.list_training_jobs(
    NameContains="TrainBioBERT",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)
job_name   = training_jobs["TrainingJobSummaries"][0]["TrainingJobName"]
job_detail = sm_client.describe_training_job(TrainingJobName=job_name)
model_uri  = job_detail["ModelArtifacts"]["S3ModelArtifacts"]

In [ ]:
hf_model = HuggingFaceModel(
    model_data=model_uri,
    role=role,
    transformers_version="4.26",
    pytorch_version="1.13",
    py_version="py39",
    sagemaker_session=session,
)

predictor = hf_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=ENDPOINT_NAME,
)
print(f"Endpoint deployed ✅: {ENDPOINT_NAME}")

## Enable Data Capture (for Model Monitor)

In [ ]:
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{BUCKET}/model-monitor/data-capture"
)
predictor.update_data_capture_config(data_capture_config=data_capture_config)
print("Data capture enabled ✅")

## Test the endpoint

In [ ]:
import pickle

sm_client.download_file = None  # reset
import boto3 as _b3
_s3 = _b3.client("s3", region_name=REGION)
_s3.download_file(BUCKET, "symptom-data/processed/label_encoder.pkl", "label_encoder.pkl")
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

test_cases = [
    "I have severe headaches and sensitivity to light",
    "My joints are swollen and painful especially in the morning",
    "I feel very thirsty, urinate frequently and feel tired",
    "I have a persistent cough, fever and difficulty breathing",
]

for symptom in test_cases:
    response   = predictor.predict({"inputs": symptom, "parameters": {"top_k": 22}})
    label_num  = int(response[0]["label"].split("_")[1])
    disease    = le.classes_[label_num]
    score      = round(response[0]["score"], 3)
    print(f"{disease:<30} (score={score})  |  {symptom[:50]}...")

## ⚠️ Delete endpoint after use (stops billing)

In [ ]:
# Run this after the demo to avoid ongoing charges (~$0.115/hr)
sm_client_del = boto3.client("sagemaker", region_name=REGION)

try:
    sm_client_del.delete_monitoring_schedule(
        MonitoringScheduleName="symptom-monitor-schedule"
    )
    print("Monitoring schedule deleted")
except Exception:
    pass

sm_client_del.delete_endpoint(EndpointName=ENDPOINT_NAME)
print("Endpoint deleted ✅")